In [1]:
import sys
import datetime
import pandas as pd
import xgboost as xgb
import optuna
from optuna.integration import XGBoostPruningCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
    StackingClassifier,
    VotingClassifier
)

/home/carlos/Documentos/BankLoanPrediction - AML/BankLoanPredictionAlgorithm/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Allow the dataset to be loaded with both a Google Colab kernel and a local kernel

# Load the dataset with Google Colab kernel and Drive file
if 'google.colab' in sys.modules:
    # Time (aprox. 25.0s)
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive
    from google.colab import auth # type: ignore
    from oauth2client.client import GoogleCredentials

    # Authenticate the User in Google Drive
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    # Google Drive ID for public sharing of the dataset
    file_id = "1jNCvMvAPawvH8avENiwbYvQReaZOiqMH"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('processed_data.csv')

    # Reading the csv and loading it into a pandas dataframe (Use pyarrow to prevent OOM error when loading)
    df = pd.read_csv("processed_data.csv", engine="pyarrow", dtype_backend="pyarrow")

# Load the dataset with local kernel and local file
else:
    # Time (13th Gen Intel Core i5-1335U: aprox. 0.5s; AMD Ryzen AI 9 HX 370 (24) @ 5.16 GHz: aprox. 1.0s)
    # Reading the csv and loading it into a pandas dataframe (Use pyarrow to prevent OOM error when loading)
    df = pd.read_csv("../data/processed/processed_data.csv", engine="pyarrow", dtype_backend="pyarrow")

# Stop Jupyter Notebook from limiting the output
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Display dataset head
df.head()

,id,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_act_il,il_util,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,acc_open_past_24mths,bc_open_to_buy,bc_util,chargeoff_within_12_mths,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,68407277,3600,False,13.99,123.03,13,10,2,55000.0,2015-12-01,0,2,38,5.91,0,2003-08-01,675,1,30,999,7,0,2765.0,29.7,13.0,False,30,False,0,722.0,144904.0,2,36.0,3,722.0,34.0,9300.0,3,1,4.0,1506.0,37.2,0,148,128,3,3,1,4,69,4,69,2,2,4,2,5,3,4,9,4,7,76.9,0.0,0,0,178050.0,7746.0,2400.0,13734.0
1,68355089,24700,False,11.99,820.28,10,10,2,65000.0,2015-12-01,0,11,41,16.06,1,1999-12-01,715,4,6,999,22,0,21470.0,19.2,38.0,False,999,False,0,0.0,204396.0,1,73.0,3,6472.0,29.0,111800.0,0,0,4.0,57830.0,27.1,0,113,192,2,2,4,2,999,0,6,0,5,5,13,17,6,20,27,5,22,97.4,7.7,0,0,314017.0,39475.0,79300.0,24667.0
2,68341763,20000,True,10.78,432.66,8,10,2,63000.0,2015-12-01,0,4,14,10.78,0,2000-08-01,695,0,999,999,6,0,7869.0,56.2,18.0,False,999,True,0,0.0,189699.0,1,73.0,2,2081.0,65.0,14000.0,2,5,6.0,2737.0,55.9,0,125,184,14,14,5,101,999,10,999,0,2,3,2,4,6,4,7,3,6,100.0,50.0,0,0,218418.0,18696.0,6200.0,14877.0
3,68476807,10400,True,22.45,289.91,25,3,2,104433.0,2015-12-01,0,6,38,25.37,1,1998-06-01,695,3,12,999,12,0,21929.0,64.5,35.0,False,999,False,0,0.0,331730.0,3,84.0,7,9702.0,78.0,34000.0,2,1,10.0,4567.0,77.5,0,128,210,4,4,6,4,12,1,12,0,4,6,5,9,10,7,19,6,12,96.6,60.0,0,0,439570.0,95768.0,20300.0,88097.0
4,68426831,11950,False,13.44,405.18,12,4,1,34000.0,2015-12-01,0,2,10,10.2,0,1987-10-01,690,0,999,999,5,0,8822.0,68.4,6.0,False,999,False,0,0.0,12798.0,1,99.0,0,4522.0,76.0,12900.0,0,0,0.0,844.0,91.0,0,338,54,32,32,0,36,999,999,999,0,2,3,2,2,2,4,4,3,5,100.0,100.0,0,0,16900.0,12798.0,9400.0,4000.0


In [3]:
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

# 'list(range())' is more efficient than list comprehension
indexes = list(range(len(X)))

# 1. First split: 70% for Training, 30% for Temp (Eval + Test)
# We use stratify=y to maintain the proportion of the 5 classes
X_train, X_temp, y_train, y_temp, train_indexes, temp_indexes = train_test_split(
    X, y, indexes, test_size=0.30, random_state=42, stratify=y
)

# 2. Second split: Divide the 30% Temp into half (15% Eval, 15% Test of total)
X_eval, X_test, y_eval, y_test, eval_indexes, test_indexes = train_test_split(
    X_temp, y_temp, temp_indexes, test_size=0.50, random_state=42, stratify=y_temp
)

# --- Code for saving indexes (when Enetz organizes folders) ---
"""
with open("train_indexes.txt", "w") as f:
    for index in train_indexes:
        f.write(f"{index}\\n")

with open("eval_indexes.txt", "w") as f:
    for index in eval_indexes:
        f.write(f"{index}\\n")
        
with open("test_indexes.txt", "w") as f:
    for index in test_indexes:
        f.write(f"{index}\\n")
"""

'\nwith open("train_indexes.txt", "w") as f:\n    for index in train_indexes:\n        f.write(f"{index}\\n")\n\nwith open("eval_indexes.txt", "w") as f:\n    for index in eval_indexes:\n        f.write(f"{index}\\n")\n\nwith open("test_indexes.txt", "w") as f:\n    for index in test_indexes:\n        f.write(f"{index}\\n")\n'

In [4]:
# =====================================================================
# 0. DATETIME HANDLING (Execute this before anything else)
# =====================================================================
def convert_dates_to_numeric(df):
    # Create a copy to avoid SettingWithCopyWarning
    df_clean = df.copy()
    
    for col in df_clean.columns:
        # Check if the column type is datetime or contains date objects
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]) or \
           isinstance(df_clean[col].dropna().iloc[0], datetime.date):
            
            # Convert to pandas datetime just in case
            df_clean[col] = pd.to_datetime(df_clean[col])
            
            # Extract Year and Month as new numeric features
            df_clean[f"{col}_year"] = df_clean[col].dt.year
            df_clean[f"{col}_month"] = df_clean[col].dt.month
            
            # Drop the original date column
            df_clean = df_clean.drop(columns=[col])
            
    return df_clean

# NOTE: Apply this to your splits once your colleague gives them to you!
X_train = convert_dates_to_numeric(X_train)
X_eval = convert_dates_to_numeric(X_eval)
X_test = convert_dates_to_numeric(X_test)


# =====================================================================
# 1. METRICS EVALUATION FUNCTION
# =====================================================================
def evaluate_model(model_name, y_true, y_pred):
    # Use 'weighted' average for multiclass metrics
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted'),
        'F1 Score': f1_score(y_true, y_pred, average='weighted'),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }
    return metrics

# =====================================================================
# 2. ENSEMBLE 1: RANDOM FOREST (BAGGING)
# =====================================================================
def objective_rf(trial):
    # Define hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'class_weight': 'balanced', # Helps with imbalanced classes internally
        'random_state': 42,
        'n_jobs': -1
    }

    # Initialize and train the model
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)

    # Predict on the evaluation set (NOT the test set)
    preds = model.predict(X_eval)
    
    # We optimize to maximize MCC (Best for multiclass imbalance)
    mcc = matthews_corrcoef(y_eval, preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting Optuna study for Random Forest...")

# Create study and run optimization
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=3)

# Example of final evaluation:
best_rf = RandomForestClassifier(**study_rf.best_params, class_weight='balanced', random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)
rf_preds = best_rf.predict(X_eval)

rf_metrics = evaluate_model("Random Forest", y_eval, rf_preds)
print(rf_metrics)

[I 2026-03-22 18:02:58,630] A new study created in memory with name: no-name-3bf68262-5387-463a-8501-2d8adfb07576


Starting Optuna study for Random Forest...


[I 2026-03-22 18:03:19,595] Trial 0 finished with value: 0.19878805714037348 and parameters: {'n_estimators': 88, 'max_depth': 11, 'min_samples_split': 15, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.19878805714037348.
[I 2026-03-22 18:04:50,009] Trial 1 finished with value: 0.24437260668253388 and parameters: {'n_estimators': 276, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.24437260668253388.
[I 2026-03-22 18:06:13,299] Trial 2 finished with value: 0.26146612039366296 and parameters: {'n_estimators': 261, 'max_depth': 25, 'min_samples_split': 6, 'min_samples_leaf': 10}. Best is trial 2 with value: 0.26146612039366296.


{'Model': 'Random Forest', 'Accuracy': 0.6981060038566188, 'Precision': 0.7451529236655496, 'Recall': 0.6981060038566188, 'F1 Score': 0.7168919965147507, 'MCC': 0.26146612039366296}


In [5]:
# =====================================================================
# ENSEMBLE 2: XGBOOST (ADVANCED BOOSTING WITH PRUNER)
# =====================================================================

def objective_xgb(trial):
    # Define hyperparameter search space
    param = {
        'objective': 'multi:softmax',
        'num_class': 5,               
        'eval_metric': 'mlogloss',    
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }

    # 1. Setup Pruning Callback monitoring the eval set
    pruning_callback = XGBoostPruningCallback(trial, 'validation_0-mlogloss')
    
    # 2. Initialize model passing callbacks (and early_stopping) HERE, not in .fit()
    model = xgb.XGBClassifier(
        **param, 
        early_stopping_rounds=10, 
        callbacks=[pruning_callback]
    )

    # 3. Train model 
    model.fit(
        X_train, y_train,
        eval_set=[(X_eval, y_eval)],
        verbose=False
    )

    # Predict on evaluation set
    preds = model.predict(X_eval)
    
    # Maximize MCC
    mcc = matthews_corrcoef(y_eval, preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting Optuna study for XGBoost with Pruner...")

# Example of final evaluation:
study_xgb = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study_xgb.optimize(objective_xgb, n_trials=3)

# Get best parameters and add required static parameters back
best_xgb_params = study_xgb.best_params
best_xgb_params['objective'] = 'multi:softmax'
best_xgb_params['num_class'] = 5
best_xgb_params['random_state'] = 42
best_xgb_params['n_jobs'] = -1

best_xgb = xgb.XGBClassifier(**best_xgb_params)
best_xgb.fit(X_train, y_train)
xgb_preds = best_xgb.predict(X_eval)

# Use the evaluation function from Iteration 1
xgb_metrics = evaluate_model("XGBoost", y_eval, xgb_preds)
print(xgb_metrics)

[I 2026-03-22 18:07:56,120] A new study created in memory with name: no-name-57557728-a7cb-413c-93f2-c5b33f410ecc


Starting Optuna study for XGBoost with Pruner...


[I 2026-03-22 18:08:36,343] Trial 0 finished with value: 0.1817823486591503 and parameters: {'n_estimators': 260, 'max_depth': 5, 'learning_rate': 0.06720952859106652, 'subsample': 0.6047346903047147, 'colsample_bytree': 0.7321303628277216}. Best is trial 0 with value: 0.1817823486591503.
[I 2026-03-22 18:08:56,205] Trial 1 finished with value: 0.15088317365073042 and parameters: {'n_estimators': 114, 'max_depth': 7, 'learning_rate': 0.03347911793358525, 'subsample': 0.7451131580464121, 'colsample_bytree': 0.809489116385951}. Best is trial 0 with value: 0.1817823486591503.
[I 2026-03-22 18:09:12,893] Trial 2 finished with value: 0.19199130533610115 and parameters: {'n_estimators': 162, 'max_depth': 6, 'learning_rate': 0.2881126573628112, 'subsample': 0.8521265784522459, 'colsample_bytree': 0.945492970275664}. Best is trial 2 with value: 0.19199130533610115.


{'Model': 'XGBoost', 'Accuracy': 0.7875186669050875, 'Precision': 0.7349957806151279, 'Recall': 0.7875186669050875, 'F1 Score': 0.7277946581271681, 'MCC': 0.1961032320825057}


In [6]:
# =====================================================================
# ENSEMBLE 3: BAGGING CLASSIFIER 
# =====================================================================

def objective_bagging_fast(trial):
    # 1. Base estimator: Decision Tree
    tree_depth = trial.suggest_int('max_depth', 3, 7)
    base_tree = DecisionTreeClassifier(max_depth=tree_depth, random_state=42)

    # 2. Bagging params optimized for low RAM and high speed
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 50),
        'max_samples': trial.suggest_float('max_samples', 0.1, 0.3), 
        'max_features': trial.suggest_float('max_features', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }

    # Initialize Bagging Classifier (use 'estimator', or 'base_estimator' for older sklearn)
    model = BaggingClassifier(estimator=base_tree, **param)
    
    # Train model
    model.fit(X_train, y_train)

    # Predict on validation set
    preds = model.predict(X_eval)
    
    # Maximize MCC
    mcc = matthews_corrcoef(y_eval, preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting lightweight Optuna study for Bagging Classifier...")

study_bagging = optuna.create_study(direction='maximize')
# Only 10 trials to keep it short and sweet
study_bagging.optimize(objective_bagging_fast, n_trials=3) 

# Example of final evaluation extraction:
# Extract best parameters but isolate max_depth for the base estimator
best_params = study_bagging.best_params
best_depth = best_params.pop('max_depth')

final_base_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_bagging = BaggingClassifier(
    estimator=final_base_tree, 
    **best_params, 
    random_state=42, 
    n_jobs=2
)

best_bagging.fit(X_train, y_train)
bagging_preds = best_bagging.predict(X_eval)

bagging_metrics = evaluate_model("Bagging Classifier", y_eval, bagging_preds)
print(bagging_metrics)

[I 2026-03-22 18:10:11,972] A new study created in memory with name: no-name-8c4e835f-4849-45b3-961d-450ccb64d77a


Starting lightweight Optuna study for Bagging Classifier...


[I 2026-03-22 18:10:20,333] Trial 0 finished with value: 0.0 and parameters: {'max_depth': 5, 'n_estimators': 25, 'max_samples': 0.17141185741398127, 'max_features': 0.6417015649517384}. Best is trial 0 with value: 0.0.
[I 2026-03-22 18:10:34,287] Trial 1 finished with value: 0.11183274525993972 and parameters: {'max_depth': 7, 'n_estimators': 42, 'max_samples': 0.180072969247219, 'max_features': 0.811342593739404}. Best is trial 1 with value: 0.11183274525993972.
[I 2026-03-22 18:10:41,211] Trial 2 finished with value: 0.0 and parameters: {'max_depth': 5, 'n_estimators': 32, 'max_samples': 0.16214339584939202, 'max_features': 0.6003257136160157}. Best is trial 1 with value: 0.11183274525993972.


{'Model': 'Bagging Classifier', 'Accuracy': 0.7832174253444618, 'Precision': 0.72485122672569, 'Recall': 0.7832174253444618, 'F1 Score': 0.7004366541663746, 'MCC': 0.11183274525993972}


In [7]:
# =====================================================================
# ENSEMBLE 4: VOTING CLASSIFIER (THE ULTIMATE TEAM)
# =====================================================================
print("Configuring Voting Classifier...")

# Make sure best_rf, best_xgb, and best_bagging are exactly the final 
# trained models you extracted from the Optuna/RandomSearch studies!

voting_model = VotingClassifier(
    estimators=[
        ('RandomForest', best_rf),
        ('XGBoost', best_xgb),
        ('FastBagging', best_bagging)
    ],
    voting='hard', # 'hard' uses majority rule. Safe and effective.
    n_jobs=1       # Keeping it at 1 to protect Colab's RAM one last time
)

# Train the Voting Classifier
# (Note: It will internally re-fit the models on X_train)
print("Training Voting Classifier (This might take a minute)...")
voting_model.fit(X_train, y_train)

# Predict on validation set
print("Evaluating Voting Classifier...")
voting_preds = voting_model.predict(X_eval)

# Get metrics
voting_metrics = evaluate_model("Voting Classifier", y_eval, voting_preds)
print("Voting Classifier done!\n")

# =====================================================================
# FINAL COMPARISON TABLE
# =====================================================================
# We gather the dictionaries generated by our evaluate_model function
# and turn them into a beautiful Pandas DataFrame.

# Make sure these variables match whatever you named them in previous steps!
all_metrics = [
    rf_metrics,       # From Iteration 1
    xgb_metrics,      # From Iteration 2
    bagging_metrics,  # From Iteration 3
    voting_metrics    # From Iteration 4
]

comparison_df = pd.DataFrame(all_metrics)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON (SORTED BY MCC)")
print("="*70)
# Sort by MCC descending to see the winner at the top
comparison_df_sorted = comparison_df.sort_values(by='MCC', ascending=False)
print(comparison_df_sorted.to_string(index=False))

Configuring Voting Classifier...
Training Voting Classifier (This might take a minute)...
Evaluating Voting Classifier...
Voting Classifier done!


FINAL MODEL COMPARISON (SORTED BY MCC)
             Model  Accuracy  Precision   Recall  F1 Score      MCC
     Random Forest  0.698106   0.745153 0.698106  0.716892 0.261466
           XGBoost  0.787519   0.734996 0.787519  0.727795 0.196103
 Voting Classifier  0.786977   0.735420 0.786977  0.726531 0.191123
Bagging Classifier  0.783217   0.724851 0.783217  0.700437 0.111833
